# Проверка датасета IRMAS

Ноутбук проверяет структуру `IRMAS-TrainingData`, считает количество файлов по классам и показывает формы Mel-spectrogram и MFCC для одного аудиофайла.

In [ ]:
from pathlib import Path
import sys

import librosa
import librosa.display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import config
from src.features import compute_mel_spectrogram, compute_mfcc, load_audio

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "IRMAS-TrainingData"
DATA_DIR

## Проверка структуры и количества файлов

In [ ]:
rows = []
for class_code in config.CLASS_CODES:
    class_dir = DATA_DIR / class_code
    wav_count = len(list(class_dir.rglob("*.wav"))) if class_dir.exists() else 0
    rows.append({
        "class_code": class_code,
        "class_name": config.CLASS_NAMES[class_code],
        "directory_exists": class_dir.exists(),
        "wav_count": wav_count,
    })

counts = pd.DataFrame(rows)
counts

In [ ]:
counts.plot(kind="bar", x="class_code", y="wav_count", legend=False, figsize=(9, 4))
plt.title("IRMAS Training Data: количество wav-файлов по классам")
plt.xlabel("Класс")
plt.ylabel("Количество файлов")
plt.tight_layout()
plt.show()

## Загрузка одного аудиофайла

In [ ]:
audio_files = sorted(DATA_DIR.glob("*/*.wav"))
if not audio_files:
    raise FileNotFoundError(f"В {DATA_DIR} не найдены wav-файлы. Проверьте расположение IRMAS-TrainingData.")

sample_path = audio_files[0]
audio = load_audio(sample_path)

print("Файл:", sample_path)
print("Форма аудио:", audio.shape)
print("Длительность после crop/pad:", len(audio) / config.SAMPLE_RATE, "сек.")

plt.figure(figsize=(10, 3))
librosa.display.waveshow(audio, sr=config.SAMPLE_RATE)
plt.title(sample_path.name)
plt.tight_layout()
plt.show()

## Mel-spectrogram и MFCC

In [ ]:
mel = compute_mel_spectrogram(audio)
mfcc = compute_mfcc(audio)

print("Mel-spectrogram shape:", mel.shape)
print("MFCC shape:", mfcc.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(mel[0], origin="lower", aspect="auto", cmap="magma")
axes[0].set_title("Mel-spectrogram")
axes[0].set_xlabel("Time frames")
axes[0].set_ylabel("Mel bins")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(mfcc[0], origin="lower", aspect="auto", cmap="magma")
axes[1].set_title("MFCC")
axes[1].set_xlabel("Time frames")
axes[1].set_ylabel("MFCC coefficients")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

Вывод: после предобработки все аудиофайлы приводятся к фиксированной длительности 3 секунды. Поэтому Mel-spectrogram и MFCC имеют постоянное число временных кадров и могут быть собраны в batch для CNN.